# 02 - Outlier analysis

Objetivo: identificar clientes extremos con percentiles, IQR, transformaciones robustas e Isolation Forest. Los clientes no se eliminan.

In [ ]:
import sys
import numpy as np
import polars as pl
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from analysis.pipeline import load_transactions, build_customer_features, transform_features, detect_anomalies

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
tx = load_transactions(ROOT / "transacciones_sinteticas.csv")
features = build_customer_features(tx)
numeric = [c for c in features.columns if c != "codigo_cliente"]
features.select(numeric).describe()

In [ ]:
def iqr_bounds(column):
    q1, q3 = features[column].quantile(0.25), features[column].quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_summary = []
for column in ["transaction_count", "gross_amount", "amount_mean", "type_hhi", "type_entropy"]:
    low, high = iqr_bounds(column)
    count = features.filter((pl.col(column) < low) | (pl.col(column) > high)).height
    outlier_summary.append({"feature": column, "iqr_low": low, "iqr_high": high, "customers": count})
pl.DataFrame(outlier_summary)

In [ ]:
matrix, columns = transform_features(features, scaler="robust")
anomaly_labels = detect_anomalies(matrix, contamination=0.02)
anomalies = features.select("codigo_cliente").with_columns(pl.Series("isolation_forest_anomaly", anomaly_labels == -1))
anomalies.group_by("isolation_forest_anomaly").len()

## Decision
Los indicadores IQR e Isolation Forest se conservan como evidencia y no como filtro. Las estrategias de clustering posteriores pueden comparar escalamiento robusto y estandarizacion.